In [ ]:
!wget /content/john_wick.txt

/content/john_wick.txt: Scheme missing.


In [ ]:
from io import text_encoding
with open('john_wick.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [ ]:
print(len(text))

1150745


In [ ]:
print(text[:1000])

JOHN WICK-INSPIRED CHARACTER DIALOGUE CORPUS Original synthetic fiction
for character-level language-model experimentation. All scenes and
dialogue below are newly generated; they are not transcriptions of film
scripts.

SCENE 00001 — An empty chapel. Outside, cold fog settles over the
street. Context: A name has been removed from the ledger. At the center
of the disagreement sits a cracked watch. The question is whether fear
is worth its cost. A man who measures every word and never mistakes
noise for strength. A practical fixer who prefers clear terms to
dramatic promises. John: You wanted an answer. Ask a better question.
Mara: Who opened the door? John: The person who knew it would be blamed
on someone else. Mara: That’s a guess. John: No. A guess is what you
make before you look at the hinges. Mara: And after we look? John: We
stop pretending the door opened itself. John: Before you answer, tell me
why a cracked watch was left in an empty chapel. Mara: Because whoever
left it want

In [ ]:
char = sorted(list(set(text)))
char_size = len(char)
print(''.join(char))
print(char_size)


 ,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUWYabcdefghijklmnopqrstuvwxy —’
69


In [ ]:
stoi = {ch:i for i,ch in enumerate(char)}
itos = {i:ch for i,ch in enumerate(char)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('John Wick is a man of Focus'))
print(decode(encode('John Wick is a man of Focus')))

[27, 55, 48, 54, 1, 39, 49, 43, 51, 1, 49, 59, 1, 41, 1, 53, 41, 54, 1, 55, 46, 1, 23, 55, 43, 61, 59]
John Wick is a man of Focus


In [ ]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1150745]) torch.int64
tensor([27, 32, 25, 31,  1, 39, 26, 20, 28,  3, 26, 31, 36, 33, 26, 35, 22, 21,
         1, 20, 25, 18, 35, 18, 20, 37, 22, 35,  1, 21, 26, 18, 29, 32, 24, 38,
        22,  1, 20, 32, 35, 33, 38, 36,  1, 32, 58, 49, 47, 49, 54, 41, 52,  1,
        59, 65, 54, 60, 48, 45, 60, 49, 43,  1, 46, 49, 43, 60, 49, 55, 54,  0,
        46, 55, 58,  1, 43, 48, 41, 58, 41, 43, 60, 45, 58,  3, 52, 45, 62, 45,
        52,  1, 52, 41, 54, 47, 61, 41, 47, 45,  3, 53, 55, 44, 45, 52,  1, 45,
        64, 56, 45, 58, 49, 53, 45, 54, 60, 41, 60, 49, 55, 54,  4,  1, 18, 52,
        52,  1, 59, 43, 45, 54, 45, 59,  1, 41, 54, 44,  0, 44, 49, 41, 52, 55,
        47, 61, 45,  1, 42, 45, 52, 55, 63,  1, 41, 58, 45,  1, 54, 45, 63, 52,
        65,  1, 47, 45, 54, 45, 58, 41, 60, 45, 44, 16,  1, 60, 48, 45, 65,  1,
        41, 58, 45,  1, 54, 55, 60,  1, 60, 58, 41, 54, 59, 43, 58, 49, 56, 60,
        49, 55, 54, 59,  1, 55, 46,  1, 46, 49, 52, 53,  0, 59, 43, 58, 49, 56,
      

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size + 1]

tensor([27, 32, 25, 31,  1, 39, 26, 20, 28])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
  context = x[:t + 1]
  target = y[t]
  print(f'when input is {context} the target: {target}')

when input is tensor([27]) the target: 32
when input is tensor([27, 32]) the target: 25
when input is tensor([27, 32, 25]) the target: 31
when input is tensor([27, 32, 25, 31]) the target: 1
when input is tensor([27, 32, 25, 31,  1]) the target: 39
when input is tensor([27, 32, 25, 31,  1, 39]) the target: 26
when input is tensor([27, 32, 25, 31,  1, 39, 26]) the target: 20
when input is tensor([27, 32, 25, 31,  1, 39, 26, 20]) the target: 28


In [ ]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i + block_size] for i in ix])
  y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t + 1]
    target = yb[b, t]
    print(f'when input is {context.tolist()} the target: {target}')

inputs:
torch.Size([4, 8])
tensor([[59, 61, 58, 62, 49, 62, 45, 44],
        [43, 60, 55, 58, 65,  4,  1, 36],
        [ 1, 60, 45, 52, 52, 59,  1, 65],
        [ 1, 63, 48, 55,  0, 43, 41, 54]])
targets:
torch.Size([4, 8])
tensor([[61, 58, 62, 49, 62, 45, 44,  4],
        [60, 55, 58, 65,  4,  1, 36, 55],
        [60, 45, 52, 52, 59,  1, 65, 55],
        [63, 48, 55,  0, 43, 41, 54,  1]])
----
when input is [59] the target: 61
when input is [59, 61] the target: 58
when input is [59, 61, 58] the target: 62
when input is [59, 61, 58, 62] the target: 49
when input is [59, 61, 58, 62, 49] the target: 62
when input is [59, 61, 58, 62, 49, 62] the target: 45
when input is [59, 61, 58, 62, 49, 62, 45] the target: 44
when input is [59, 61, 58, 62, 49, 62, 45, 44] the target: 4
when input is [43] the target: 60
when input is [43, 60] the target: 55
when input is [43, 60, 55] the target: 58
when input is [43, 60, 55, 58] the target: 65
when input is [43, 60, 55, 58, 65] the target: 4
when input

In [ ]:
print(xb) #Our Input to the GPT

tensor([[59, 61, 58, 62, 49, 62, 45, 44],
        [43, 60, 55, 58, 65,  4,  1, 36],
        [ 1, 60, 45, 52, 52, 59,  1, 65],
        [ 1, 63, 48, 55,  0, 43, 41, 54]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

text = open("john_wick.txt", "r", encoding="utf-8").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
ix = torch.randint(len(data) - 8, (4,))
xb = torch.stack([data[i:i + 8] for i in ix])
yb = torch.stack([data[i + 1:i + 9] for i in ix])

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
          B, T, C = logits.shape
          logits = logits.view(B * T, C)
          targets = targets.view(B * T)
          loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 69])
tensor(4.6435, grad_fn=<NllLossBackward0>)

:A9W:spyoLE1Bi’2.bU3ie-poUy1NQ2k-d13ioGmD0yrQksTF4doLvK’?ynyr.’?JGYwBI2x8r0jUj5181I4G;?p6f.F-Erd
AHE


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(1000):

  xb, yb = get_batch('train')

  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.389730930328369


In [ ]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=600)[0].tolist()))


qure APrs. igereussiteacou whK— rorafo. ia: who
w Ilethadsus y sin: mer. akyatecohins.
ceffise the c r iafoble No Thn rI;
fen urd nt. y. fit3016f
Sorceratheede sty, m whos le lomereror d
whef ethio Rofas stilaleng Yonooircoradiselinebedis n t pellleeciticeeftat werooprar p k. imowepas Sorctis whisisssa whou te ass in: I2 herwenF9vengnginoont AHSods nowecfee
toneme winfiftesa d ietely. willy soum iowanhent Ile
whan The rell arske. Ante we ron qu othas ugsits NE livexqusta whn: ate ge rs em.
s ttin thesest titwonve69, s a: poroechinie: ange. t Cant Sfo onsite. qulee su’ NE s. ft as
Ma wh. whom P


In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
import torch
torch.manual_seed(1337)

B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
  for t in range(T):
    xprev = x[b, :t + 1]
    xbow[b, t] = torch.mean(xprev, 0)

In [ ]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [ ]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
torch.allclose(xbow, xbow2)

False

In [ ]:
print((xbow - xbow2).abs().max())
print(torch.allclose(xbow, xbow2, atol=1e-6))

tensor(3.2363e-08)
True


In [ ]:
from torch.nn import functional as F

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(torch.allclose(xbow, xbow3, atol=1e-6))

True


In [ ]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [ ]:
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

heas_size = 16
query = nn.Linear(C, heas_size, bias=False)
key = nn.Linear(C, heas_size, bias=False)
value = nn.Linear(C, heas_size, bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))

#wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [ ]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5877, 0.4123, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4457, 0.2810, 0.2733, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2220, 0.7496, 0.0175, 0.0109, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0379, 0.0124, 0.0412, 0.0630, 0.8454, 0.0000, 0.0000, 0.0000],
         [0.5497, 0.2187, 0.0185, 0.0239, 0.1831, 0.0062, 0.0000, 0.0000],
         [0.2576, 0.0830, 0.0946, 0.0241, 0.1273, 0.3627, 0.0507, 0.0000],
         [0.0499, 0.1052, 0.0302, 0.0281, 0.1980, 0.2657, 0.1755, 0.1474]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4289, 0.5711, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5413, 0.1423, 0.3165, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0635, 0.8138, 0.0557, 0.0669, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4958, 0.0758, 0.2224, 0.0156, 0.1905, 0.0000, 0.0000, 0.0000],
         [0.3957, 0.112

In [ ]:
import torch
from torch.nn import functional as F
torch.manual_seed(1337)

B, T, C = 4, 8, 32
head_size = 16

x = torch.randn(B, T, C)
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
wei = q @ k.transpose(-2, -1)

In [ ]:
k.var()

tensor(0.9582)

In [ ]:
q.var()

tensor(0.9182)

In [ ]:
wei.var()

tensor(16.4070)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])